# StaySure Hotel Cancellation ML
## Milestone 2 — Data Cleaning and Leakage Control

**Business goal:** prepare a trustworthy modeling table that can predict whether a hotel reservation will be canceled using information available at the time of booking.

This notebook is intentionally **learner-led**. It provides explanations, examples, checkpoints, and reusable code, but you must make and document four important decisions before saving the cleaned data.

### Deliverables

- `data/processed/hotel_bookings_clean.csv` — local modeling table (do not commit the CSV)
- `reports/milestone_02_cleaning_metrics.csv` — before/after quality metrics
- `reports/milestone_02_cleaning_decisions.csv` — documented cleaning decisions

> Never modify `data/raw/hotel_bookings.csv`. The raw file is your reproducible source of truth.


### What you should be able to explain afterward

1. Why cleaning rules must follow the business prediction point.
2. Why exact duplicate rows are not automatically errors when no booking ID exists.
3. How you handled each missing-value pattern.
4. Which columns would leak future information into the model.
5. How cleaning changed the row count and cancellation rate.

### Milestone workflow

1. Load and verify the raw data.
2. Investigate duplicates rather than silently deleting them.
3. Define missing-value rules.
4. remove invalid records using explicit business rules.
5. Remove leakage and timing-sensitive fields.
6. Validate, document, and save the result.


## 1. Setup


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("pandas version:", pd.__version__)


pandas version: 3.0.6


The next cell locates the project root whether the notebook is launched from the repository root or from the `notebooks/` folder.


In [2]:
cwd = Path.cwd().resolve()

if (cwd / "data" / "raw" / "hotel_bookings.csv").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw" / "hotel_bookings.csv").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find data/raw/hotel_bookings.csv. "
        "Open VS Code at the project root and confirm the CSV is in data/raw/."
    )

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "hotel_bookings.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH)


Project root: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml
Raw data: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\data\raw\hotel_bookings.csv


## 2. Load and verify the raw dataset


In [3]:
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
display(df_raw.head())


Raw shape: 119,390 rows x 32 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
# Data-contract checks protect the project from using the wrong file or schema.
required_columns = {
    "hotel", "is_canceled", "lead_time", "arrival_date_year",
    "adults", "children", "babies", "adr",
    "reservation_status", "reservation_status_date"
}

missing_required = required_columns.difference(df_raw.columns)
assert not missing_required, f"Missing required columns: {sorted(missing_required)}"
assert set(df_raw["is_canceled"].dropna().unique()).issubset({0, 1})

print("Data-contract checks passed.")


Data-contract checks passed.


In [5]:
starting_profile = pd.Series({
    "rows": len(df_raw),
    "columns": df_raw.shape[1],
    "exact_duplicate_rows_removed_if_drop_duplicates": int(df_raw.duplicated().sum()),
    "missing_cells": int(df_raw.isna().sum().sum()),
    "cancellation_rate_percent": round(df_raw["is_canceled"].mean() * 100, 2),
}, name="value").to_frame()

display(starting_profile)


,value
rows,119390.00
columns,32.00
exact_duplicate_rows_removed_if_drop_duplicates,31994.00
missing_cells,129425.00
cancellation_rate_percent,37.04


## 3. Decision 1 — Exact duplicates

`drop_duplicates()` is easy to write, but the business interpretation matters more than the syntax.

This dataset has no unique booking ID. Two identical rows could therefore be:

- accidental duplicate records, **or**
- separate customers whose recorded booking details happen to be identical.

Deleting duplicates may also change the target distribution. Measure that impact before choosing a policy.


In [6]:
duplicate_mask_all = df_raw.duplicated(keep=False)
duplicate_rows_to_remove = df_raw.duplicated(keep="first")
df_deduplicated_preview = df_raw.drop_duplicates()

duplicate_comparison = pd.DataFrame({
    "version": ["raw", "drop_duplicates preview", "rows that would be removed"],
    "rows": [len(df_raw), len(df_deduplicated_preview), int(duplicate_rows_to_remove.sum())],
    "cancellation_rate_percent": [
        df_raw["is_canceled"].mean() * 100,
        df_deduplicated_preview["is_canceled"].mean() * 100,
        df_raw.loc[duplicate_rows_to_remove, "is_canceled"].mean() * 100,
    ],
}).round(2)

display(duplicate_comparison)


,version,rows,cancellation_rate_percent
0,raw,119390,37.04
1,drop_duplicates preview,87396,27.49
2,rows that would be removed,31994,63.13


### TODO 1 — Choose and explain a duplicate policy

Set `DUPLICATE_POLICY` to one of these values:

- `"keep"` — recommended for the primary dataset because there is no booking ID and removing rows materially changes the cancellation rate.
- `"drop"` — acceptable only if you clearly state that you are assuming exact matches are accidental duplicate records.

Write 2–4 sentences in the Markdown cell below explaining your choice. Mention both the missing booking ID and the target-rate change.


In [7]:
# TODO 1: Replace None with "keep" or "drop".
DUPLICATE_POLICY = "keep"

if DUPLICATE_POLICY not in {"keep", "drop", None}:
    raise ValueError('DUPLICATE_POLICY must be "keep" or "drop".')

print("Duplicate policy:", DUPLICATE_POLICY or "TODO — choose keep or drop")


Duplicate policy: keep


**TODO 1 explanation:**  
Write your duplicate-policy explanation here.


## DUPLICATE POLICY Decision 
I chose to keep the exact duplicate rows because the dataset does not include a unique booking ID. Identical rows may represent separate reservations with the same recorded characteristics rather than data-entry errors. Removing them would materially change the cancellation rate from approximately 37.04% to 27.49%, so keeping them better preserves the observed booking distribution. I will later compare results against a deduplicated version as a sensitivity check.

## 4. Decision 2 — Missing values


In [8]:
missing_summary = (
    df_raw.isna().sum()
    .loc[lambda s: s.gt(0)]
    .sort_values(ascending=False)
    .to_frame("missing_count")
)
missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(df_raw) * 100
).round(2)

display(missing_summary)


,missing_count,missing_percent
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


### Proposed missing-value rules

| Column | Proposed rule | Reason |
|---|---|---|
| `children` | Fill with `0` | Only four values are missing; zero is a simple, explainable assumption. |
| `country` | Fill with `"Unknown"` | Preserve the bookings and make missingness explicit. |
| `agent` | Fill with `0` | Treat zero as “no recorded agent”; this field acts like a category code. |
| `company` | Drop the column | More than 90% is missing, so it contributes little reliable information. |

The first three transformations are provided. You decide whether to drop `company`.


In [9]:
# TODO 2: Replace None with True or False.
# Recommended: True because company is more than 90% missing.
DROP_COMPANY = True

if DROP_COMPANY not in {True, False, None}:
    raise ValueError("DROP_COMPANY must be True or False.")

print("Drop company:", DROP_COMPANY if DROP_COMPANY is not None else "TODO")


Drop company: True


**TODO 2 explanation:**  
Explain why you chose to keep or drop `company`. Include its approximate missing percentage.


## DROP_COMPANY Decision
I chose to drop the company column because 112,593 of its 119,390 values—approximately 94.31%—are missing. Filling such a large amount of missing data would introduce assumptions without providing much reliable information. Removing the column keeps the baseline model simpler and easier to explain.

## 5. Decision 3 — Invalid or out-of-scope records


In [10]:
children_for_check = df_raw["children"].fillna(0)
total_guests_for_check = df_raw["adults"] + children_for_check + df_raw["babies"]
total_nights_for_check = (
    df_raw["stays_in_weekend_nights"] + df_raw["stays_in_week_nights"]
)

invalid_summary = pd.Series({
    "negative_adr": int((df_raw["adr"] < 0).sum()),
    "zero_total_guests": int((total_guests_for_check == 0).sum()),
    "zero_total_nights": int((total_nights_for_check == 0).sum()),
}, name="row_count").to_frame()

display(invalid_summary)


,row_count
negative_adr,1
zero_total_guests,180
zero_total_nights,715


For this project, the proposed scope is a reservation with:

- a non-negative average daily rate (`adr >= 0`),
- at least one guest, and
- at least one hotel night.

Removing these records is a **business-scope rule**, not a claim that every row is fraudulent or impossible.


In [11]:
# TODO 3: Replace None with True or False.
# Recommended: True for a clear hotel-stay prediction scope.
REMOVE_INVALID_ROWS = True

if REMOVE_INVALID_ROWS not in {True, False, None}:
    raise ValueError("REMOVE_INVALID_ROWS must be True or False.")

print(
    "Remove invalid/out-of-scope rows:",
    REMOVE_INVALID_ROWS if REMOVE_INVALID_ROWS is not None else "TODO"
)


Remove invalid/out-of-scope rows: True


**TODO 3 explanation:**  
Explain whether you will remove the three categories above and why they do or do not fit the project’s business scope.


## REMOVE_INVALID_ROWS Decision
I chose to remove the invalid or out-of-scope rows because this project is intended to predict cancellations for legitimate overnight hotel stays. Reservations with a negative average daily rate, no recorded guests, or zero total nights do not meet that business definition and could introduce noise into the model. These records are being removed using documented and repeatable business rules rather than being silently deleted.

## 6. Decision 4 — Leakage and prediction timing


The prediction point is **when the reservation is booked**.

- Confirmed leakage: `reservation_status`, `reservation_status_date` reveal the eventual outcome or information recorded afterward.
- Timing-sensitive fields: `assigned_room_type`, `booking_changes`, and `days_in_waiting_list` may be created or updated after the original booking.

Confirmed leakage will always be removed. For a strict booking-time model, the recommended choice is also to remove all three timing-sensitive fields.


In [12]:
CONFIRMED_LEAKAGE = ["reservation_status", "reservation_status_date"]
TIMING_SENSITIVE = ["assigned_room_type", "booking_changes", "days_in_waiting_list"]

# TODO 4: Replace None with True or False.
# Recommended: True for a strict booking-time prediction model.
DROP_TIMING_SENSITIVE = True

if DROP_TIMING_SENSITIVE not in {True, False, None}:
    raise ValueError("DROP_TIMING_SENSITIVE must be True or False.")

print("Confirmed leakage to drop:", CONFIRMED_LEAKAGE)
print(
    "Drop timing-sensitive fields:",
    DROP_TIMING_SENSITIVE if DROP_TIMING_SENSITIVE is not None else "TODO"
)


Confirmed leakage to drop: ['reservation_status', 'reservation_status_date']
Drop timing-sensitive fields: True


**TODO 4 explanation:**  
Explain why future information can make validation scores look excellent while making the deployed model unreliable.


## DROP_TIMING_SENSITIVE Decision
I chose to remove the timing-sensitive fields because the model is intended to make predictions when a reservation is first booked. Features such as assigned_room_type, booking_changes, and days_in_waiting_list may be created or updated after that prediction point. Including future information could artificially improve validation scores, but the same information would not be available when the deployed model makes a real prediction, making its performance unreliable.

## 7. Build a reusable cleaning function


This function makes the cleaning policy repeatable. Notice that it receives explicit decisions rather than hiding them inside scattered notebook cells.


In [13]:
def clean_hotel_bookings(
    raw_df,
    duplicate_policy,
    drop_company,
    remove_invalid_rows,
    drop_timing_sensitive,
):
    'Return a cleaned copy of the hotel-bookings dataset.'
    cleaned = raw_df.copy()

    # 1. Duplicate policy
    if duplicate_policy == "drop":
        cleaned = cleaned.drop_duplicates().copy()
    elif duplicate_policy != "keep":
        raise ValueError('duplicate_policy must be "keep" or "drop"')

    # 2. Missing values
    cleaned["children"] = cleaned["children"].fillna(0)
    cleaned["country"] = cleaned["country"].fillna("Unknown")
    cleaned["agent"] = cleaned["agent"].fillna(0)

    if drop_company:
        cleaned = cleaned.drop(columns=["company"])
    else:
        cleaned["company"] = cleaned["company"].fillna(0)

    # 3. Business-scope validation
    cleaned["total_guests"] = (
        cleaned["adults"] + cleaned["children"] + cleaned["babies"]
    )
    cleaned["total_nights"] = (
        cleaned["stays_in_weekend_nights"] + cleaned["stays_in_week_nights"]
    )

    if remove_invalid_rows:
        valid_mask = (
            cleaned["adr"].ge(0)
            & cleaned["total_guests"].gt(0)
            & cleaned["total_nights"].gt(0)
        )
        cleaned = cleaned.loc[valid_mask].copy()

    # 4. Leakage control
    columns_to_drop = list(CONFIRMED_LEAKAGE)
    if drop_timing_sensitive:
        columns_to_drop.extend(TIMING_SENSITIVE)

    cleaned = cleaned.drop(columns=columns_to_drop, errors="ignore")
    cleaned = cleaned.reset_index(drop=True)
    return cleaned


### Run the function after completing TODOs 1–4


In [14]:
decision_values = {
    "DUPLICATE_POLICY": DUPLICATE_POLICY,
    "DROP_COMPANY": DROP_COMPANY,
    "REMOVE_INVALID_ROWS": REMOVE_INVALID_ROWS,
    "DROP_TIMING_SENSITIVE": DROP_TIMING_SENSITIVE,
}
unfinished_decisions = [name for name, value in decision_values.items() if value is None]

if unfinished_decisions:
    df_clean = None
    print("Complete these decisions first:", unfinished_decisions)
else:
    df_clean = clean_hotel_bookings(
        df_raw,
        duplicate_policy=DUPLICATE_POLICY,
        drop_company=DROP_COMPANY,
        remove_invalid_rows=REMOVE_INVALID_ROWS,
        drop_timing_sensitive=DROP_TIMING_SENSITIVE,
    )
    print(f"Clean shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
    display(df_clean.head())


Clean shape: 118,564 rows x 28 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_guests,total_nights
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,No Deposit,0.0,Transient,75.0,0,0,1.0,1
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,No Deposit,304.0,Transient,75.0,0,0,1.0,1
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,No Deposit,240.0,Transient,98.0,0,1,2.0,2
3,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,No Deposit,240.0,Transient,98.0,0,1,2.0,2
4,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,No Deposit,0.0,Transient,107.0,0,0,2.0,2


## 8. Validate the cleaned table


In [15]:
if df_clean is None:
    print("Validation is waiting for TODOs 1–4.")
else:
    validation_checks = pd.Series({
        "target_has_no_missing_values": df_clean["is_canceled"].notna().all(),
        "target_is_binary": set(df_clean["is_canceled"].unique()).issubset({0, 1}),
        "children_has_no_missing_values": df_clean["children"].notna().all(),
        "country_has_no_missing_values": df_clean["country"].notna().all(),
        "agent_has_no_missing_values": df_clean["agent"].notna().all(),
        "confirmed_leakage_removed": not any(
            column in df_clean.columns for column in CONFIRMED_LEAKAGE
        ),
        "row_count_is_not_larger_than_raw": len(df_clean) <= len(df_raw),
    }, name="passed")
    display(validation_checks.to_frame())


,passed
target_has_no_missing_values,True
target_is_binary,True
children_has_no_missing_values,True
country_has_no_missing_values,True
agent_has_no_missing_values,True
confirmed_leakage_removed,True
row_count_is_not_larger_than_raw,True


In [16]:
if df_clean is None:
    print("Before/after comparison is waiting for TODOs 1–4.")
else:
    before_after = pd.DataFrame({
        "metric": ["rows", "columns", "missing_cells", "cancellation_rate_percent"],
        "before": [
            len(df_raw),
            df_raw.shape[1],
            int(df_raw.isna().sum().sum()),
            round(df_raw["is_canceled"].mean() * 100, 2),
        ],
        "after": [
            len(df_clean),
            df_clean.shape[1],
            int(df_clean.isna().sum().sum()),
            round(df_clean["is_canceled"].mean() * 100, 2),
        ],
    })
    display(before_after)


,metric,before,after
0,rows,119390.00,118564.00
1,columns,32.00,28.00
2,missing_cells,129425.00,0.00
3,cancellation_rate_percent,37.04,37.26


### TODO 5 — Interpret the before/after comparison

Write 3–5 sentences below:

1. How many rows were removed, if any?
2. Did the cancellation rate change materially?
3. Why does that change matter for modeling and evaluation?
4. What information was intentionally removed to prevent leakage?


**TODO 5 interpretation:**  
Write your before/after interpretation here.


## Data Cleaning Summary
The cleaning process removed 826 invalid or out-of-scope records, reducing the dataset from 119,390 to 118,564 rows. The cancellation rate increased slightly from 37.04% to 37.26%, a change of only 0.22 percentage points, so the cleaning rules did not materially alter the target distribution. This is important because the cleaned dataset still represents approximately the same business population as the raw data. To prevent leakage, I removed reservation_status, reservation_status_date, assigned_room_type, booking_changes, and days_in_waiting_list, because they contain or may contain information unavailable when the reservation is initially booked.

## 9. Create the cleaning audit trail


In [17]:
if df_clean is None:
    cleaning_metrics = None
    cleaning_decisions = None
    print("Audit tables are waiting for TODOs 1–4.")
else:
    cleaning_metrics = pd.DataFrame({
        "metric": [
            "raw_rows", "clean_rows", "rows_removed", "raw_columns",
            "clean_columns", "raw_missing_cells", "clean_missing_cells",
            "raw_cancellation_rate_percent", "clean_cancellation_rate_percent",
        ],
        "value": [
            len(df_raw), len(df_clean), len(df_raw) - len(df_clean),
            df_raw.shape[1], df_clean.shape[1],
            int(df_raw.isna().sum().sum()), int(df_clean.isna().sum().sum()),
            round(df_raw["is_canceled"].mean() * 100, 2),
            round(df_clean["is_canceled"].mean() * 100, 2),
        ],
    })

    cleaning_decisions = pd.DataFrame([
        {
            "area": "exact duplicates",
            "decision": DUPLICATE_POLICY,
            "reason": "No booking ID; measure target-distribution impact before removal.",
        },
        {
            "area": "company missingness",
            "decision": "drop" if DROP_COMPANY else "fill with 0",
            "reason": "Company has very high missingness.",
        },
        {
            "area": "invalid/out-of-scope rows",
            "decision": "remove" if REMOVE_INVALID_ROWS else "keep",
            "reason": "Defines valid hotel stays by ADR, guest count, and night count.",
        },
        {
            "area": "confirmed leakage",
            "decision": "drop",
            "reason": ", ".join(CONFIRMED_LEAKAGE),
        },
        {
            "area": "timing-sensitive features",
            "decision": "drop" if DROP_TIMING_SENSITIVE else "keep",
            "reason": ", ".join(TIMING_SENSITIVE),
        },
    ])

    display(cleaning_metrics)
    display(cleaning_decisions)


,metric,value
0,raw_rows,119390.00
1,clean_rows,118564.00
2,rows_removed,826.00
3,raw_columns,32.00
4,clean_columns,28.00
5,raw_missing_cells,129425.00
6,clean_missing_cells,0.00
7,raw_cancellation_rate_percent,37.04
8,clean_cancellation_rate_percent,37.26


,area,decision,reason
0,exact duplicates,keep,No booking ID; measure target-distribution imp...
1,company missingness,drop,Company has very high missingness.
2,invalid/out-of-scope rows,remove,"Defines valid hotel stays by ADR, guest count,..."
3,confirmed leakage,drop,"reservation_status, reservation_status_date"
4,timing-sensitive features,drop,"assigned_room_type, booking_changes, days_in_w..."


## 10. Save the cleaned data and reports


Before saving:

1. Complete TODOs 1–5.
2. Run the notebook from top to bottom without errors.
3. Confirm every validation check is `True`.
4. Change `READY_TO_SAVE` to `True`.

The processed CSV is reproducible and should remain local. Add this line to the project-root `.gitignore`:

```text
data/processed/*.csv
```

The two small report CSV files **should** be committed.


In [18]:
# TODO 6: Change to True only after reviewing all decisions and checks.
READY_TO_SAVE = True

if not READY_TO_SAVE:
    print("Nothing saved. Complete the notebook, then set READY_TO_SAVE = True.")
elif df_clean is None:
    print("Nothing saved. Complete TODOs 1–4 and rerun the cleaning function.")
else:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)

    clean_data_path = PROCESSED_DIR / "hotel_bookings_clean.csv"
    metrics_path = REPORTS_DIR / "milestone_02_cleaning_metrics.csv"
    decisions_path = REPORTS_DIR / "milestone_02_cleaning_decisions.csv"

    df_clean.to_csv(clean_data_path, index=False)
    cleaning_metrics.to_csv(metrics_path, index=False)
    cleaning_decisions.to_csv(decisions_path, index=False)

    print("Saved:")
    print(" -", clean_data_path)
    print(" -", metrics_path)
    print(" -", decisions_path)


Saved:
 - C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\data\processed\hotel_bookings_clean.csv
 - C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\reports\milestone_02_cleaning_metrics.csv
 - C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\reports\milestone_02_cleaning_decisions.csv


In [20]:
# Check whether the processed-data ignore rule is present.
gitignore_path = PROJECT_ROOT / ".gitignore"
required_ignore_rule = "data/processed/*.csv"

if not gitignore_path.exists():
    print("TODO: Create .gitignore at the project root and add:", required_ignore_rule)
else:
    gitignore_text = gitignore_path.read_text(encoding="utf-8")
    if required_ignore_rule in gitignore_text:
        print("Processed-data ignore rule found.")
    else:
        print("TODO: Add this line to .gitignore:", required_ignore_rule)


Processed-data ignore rule found.


## Interview-ready explanation

Practice explaining the milestone in about 60–90 seconds:

> “I preserved the raw hotel-booking data and built a reproducible cleaning function. I investigated exact duplicates before deciding what to do because the dataset has no booking ID and deduplication changes the target rate. I handled missing values according to their meaning and removed fields that reveal the eventual reservation outcome. I also defined valid hotel-stay rules, measured the effect of every transformation, and exported a cleaning audit trail.”

### Questions you should be ready to answer

1. Why did you keep or remove exact duplicate rows?
2. Why is `reservation_status` target leakage?
3. Why did you drop or retain `company`?
4. What is the difference between an invalid value and an outlier?
5. How did cleaning change the class balance?


## Data Cleaning Detailed Answers

1. Why did you keep or remove exact duplicate rows?
I kept the exact duplicate rows because this dataset does not contain a unique booking identification number. Therefore, two rows with identical values could represent two legitimate reservations rather than an accidental duplicate. Removing all 31,994 matching rows would reduce the cancellation rate from approximately 37.04% to 27.49%, which is a significant change in the target distribution. This suggests that the duplicate-looking rows are not randomly distributed. I decided to retain them for the initial model and document the decision rather than remove potentially valid bookings without sufficient evidence.
2. Why is reservation_status target leakage?
reservation_status is target leakage because it directly reveals the final outcome of a reservation. For example, values such as “Canceled” or “Check-Out” tell the model whether the booking was canceled—the exact result represented by the is_canceled target variable. This information would not be available when the reservation is first created, which is when the model is supposed to make its prediction. Including it would produce unrealistically strong model performance that would not work in a real-world deployment.
3. Why did you drop or retain company?
I dropped the company column because approximately 94.31% of its values were missing. With so little information available, the column would likely contribute limited predictive value while adding unnecessary complexity to the preprocessing pipeline. Although the presence or absence of a company could potentially contain useful information, the column is too incomplete for this straightforward first version of the project. I documented the decision so it could be reconsidered during a future experiment.
4. What is the difference between an invalid value and an outlier?
An invalid value violates the dataset’s business rules or represents an impossible record, such as a negative average daily rate, a reservation with zero guests, or a booking with zero nights. An outlier is unusual but may still be legitimate, such as a very high room rate or an unusually long lead time. Invalid values may need to be corrected or removed, while outliers should first be investigated before deciding whether to retain, transform, cap, or remove them. An extreme value should not automatically be treated as an error.
5. How did cleaning change the class balance?
Cleaning removed 826 invalid or out-of-scope records, reducing the dataset from 119,390 to 118,564 rows. The cancellation rate increased slightly from 37.04% to 37.26%, a difference of only 0.22 percentage points. This is a negligible change, meaning the cleaning process did not materially distort the target distribution. The cleaned dataset therefore continues to represent approximately the same business population as the original dataset.

## Short answers
1. Duplicates: I kept exact duplicates because there is no unique booking ID, so identical rows may represent legitimate separate bookings. Removing them also changed the cancellation rate significantly.
2. Target leakage: reservation_status reveals the final booking outcome, so using it would give the model information unavailable when making a real prediction.
3. Company: I dropped company because approximately 94% of its values were missing, making it unlikely to provide reliable value in this first model.
4. Invalid vs. outlier: An invalid value breaks a business rule, such as negative ADR or zero guests. An outlier is unusual but may still be a legitimate observation.
5. Class balance: Cleaning removed 826 rows, while the cancellation rate changed only from 37.04% to 37.26%. Therefore, cleaning did not meaningfully distort the target distribution.

## Milestone 2 completion checklist

- [ ] TODOs 1–6 are completed in your own words.
- [ ] The raw CSV was not modified.
- [ ] The duplicate-policy impact was measured.
- [ ] Missing-value decisions were documented.
- [ ] Invalid/out-of-scope records were handled explicitly.
- [ ] Confirmed leakage fields were removed.
- [ ] All validation checks passed.
- [ ] `data/processed/*.csv` was added to `.gitignore`.
- [ ] The cleaned CSV exists locally in `data/processed/`.
- [ ] Both Milestone 2 report CSV files exist in `reports/`.
- [ ] The notebook and reports were committed and pushed to the Milestone 2 branch.

### Files to commit when finished

```text
.gitignore
notebooks/02_data_cleaning.ipynb
reports/milestone_02_cleaning_metrics.csv
reports/milestone_02_cleaning_decisions.csv
```

Do **not** commit `data/processed/hotel_bookings_clean.csv`.
